In [2]:
import pandas as pd
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

log_reg = LogisticRegression(max_iter= 5000, random_state= 42, class_weight= 'balanced')

log_reg.fit(X_train, y_train)

predictions = log_reg.predict(X_test)
probabilities = log_reg.predict_proba(X_test)[:, 1]

print(classification_report(y_test, predictions))
print(f'ROC AUC Score: {roc_auc_score(y_test, probabilities)}')

              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1035
           1       0.50      0.79      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC AUC Score: 0.8416337285902504


In [4]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10,],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': [None, 'balanced']
    }

log_reg_grid = LogisticRegression(max_iter= 5000, random_state= 42)

gridsearch = GridSearchCV(log_reg_grid, param_grid, scoring= 'f1', cv= 5, n_jobs= -1, verbose= 2)

gridsearch.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,estimator,LogisticRegre...ndom_state=42)
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [5]:
gridsearch.best_params_

{'C': 1, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'liblinear'}

In [6]:
gridsearch.best_score_

np.float64(0.6321008739435288)

In [7]:
gridsearch.best_estimator_

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'liblinear'
,max_iter,5000
,multi_class,'deprecated'


In [8]:
best_lr_f1 = gridsearch.best_estimator_

y_pred_f1 = best_lr_f1.predict(X_test)
y_prob_f1 = best_lr_f1.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_f1))
print(f'\nTuned ROC-AUC: {roc_auc_score(y_test, y_prob_f1)}')

              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1035
           1       0.50      0.79      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409


Tuned ROC-AUC: 0.8415355602056369


In [9]:
param_grid = {
    'C': [0.01, 0.1, 1, 10,],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': [None, 'balanced']
    }

log_reg_grid = LogisticRegression(max_iter= 5000, random_state= 42)

gridsearch = GridSearchCV(log_reg_grid, param_grid, scoring= 'recall', cv= 5, n_jobs= -1, verbose= 2)

gridsearch.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,estimator,LogisticRegre...ndom_state=42)
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}"
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l1'


In [10]:
gridsearch.best_params_

{'C': 0.01, 'class_weight': 'balanced', 'penalty': 'l1', 'solver': 'liblinear'}

In [11]:
gridsearch.best_score_

np.float64(0.8187290969899665)

In [12]:
gridsearch.best_estimator_

,penalty,'l1'
,dual,False
,tol,0.0001
,C,0.01
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'liblinear'
,max_iter,5000
,multi_class,'deprecated'


In [13]:
best_lr_recall = gridsearch.best_estimator_

y_pred_recall = best_lr_recall.predict(X_test)
y_prob_recall = best_lr_recall.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_recall))
print(f'\nTuned ROC-AUC: {roc_auc_score(y_test, y_prob_recall)}')

              precision    recall  f1-score   support

           0       0.92      0.70      0.79      1035
           1       0.50      0.83      0.62       374

    accuracy                           0.73      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.81      0.73      0.75      1409


Tuned ROC-AUC: 0.8390697253868609


In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def model_comparison(name: str, y_true, model_predictions, y_prob):
    return{'Name': name, 
           'Accuracy': accuracy_score(y_true, model_predictions),
           'Precision': precision_score(y_true, model_predictions),
           'Recall': recall_score(y_true, model_predictions),
           'F1 Score': f1_score(y_true, model_predictions),
           'ROC-AUC': roc_auc_score(y_test, y_prob)}

results = []
results.append(model_comparison('Original Logistic Model', y_test, predictions, probabilities))
results.append(model_comparison('F1 GridSearch', y_test, y_pred_f1, y_prob_f1))
results.append(model_comparison('Recall GridSearch', y_test, y_pred_recall, y_prob_recall))

results_df  = pd.DataFrame(results)
results_df

,Name,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Original Logistic Model,0.735273,0.500843,0.794118,0.614271,0.841634
1,F1 GridSearch,0.735273,0.500843,0.794118,0.614271,0.841536
2,Recall GridSearch,0.733854,0.499192,0.826203,0.622356,0.839070


In [15]:
y_probabilities = best_lr_recall.predict_proba(X_test)[:, 1]

In [17]:
import numpy as np

threshold = np.arange(0.1, 0.9, 0.05)
result = []

for prob in threshold:
    y_pred_prob = (y_probabilities >= prob).astype(int)

    result.append({'threshold': prob,
                   'precision': precision_score(y_test, y_pred_prob),
                   'recall': recall_score(y_test, y_pred_prob),
                   'f1': f1_score(y_test, y_pred_prob)})
    
threshold_df = pd.DataFrame(result)
threshold_df

,threshold,precision,recall,f1
0,0.10,0.296887,0.994652,0.457283
1,0.15,0.320966,0.994652,0.485323
2,0.20,0.348684,0.991979,0.515994
3,0.25,0.370821,0.978610,0.537840
4,0.30,0.393172,0.954545,0.556942
5,0.35,0.418465,0.933155,0.577815
6,0.40,0.436856,0.906417,0.589565
7,0.45,0.468116,0.863636,0.607143
8,0.50,0.499192,0.826203,0.622356
9,0.55,0.522769,0.767380,0.621885


In [18]:
# Best F1 threshold
best_f1_row = threshold_df.loc[threshold_df['f1'].idxmax()]
best_f1_row

threshold    0.500000
precision    0.499192
recall       0.826203
f1           0.622356
Name: 8, dtype: float64

In [19]:
# Best Recall with reasonable precision
threshold_df.sort_values("recall", ascending=False).head()

,threshold,precision,recall,f1
0,0.10,0.296887,0.994652,0.457283
1,0.15,0.320966,0.994652,0.485323
2,0.20,0.348684,0.991979,0.515994
3,0.25,0.370821,0.978610,0.537840
4,0.30,0.393172,0.954545,0.556942


In [20]:
BEST_THRESHOLD = 0.35

y_pred_final = (y_probabilities >= BEST_THRESHOLD).astype(int)

print(classification_report(y_test, y_pred_final))
print("ROC-AUC:", roc_auc_score(y_test, y_probabilities))

              precision    recall  f1-score   support

           0       0.96      0.53      0.68      1035
           1       0.42      0.93      0.58       374

    accuracy                           0.64      1409
   macro avg       0.69      0.73      0.63      1409
weighted avg       0.81      0.64      0.66      1409

ROC-AUC: 0.8390697253868609
